# Q3: when to switch a signal off

Revised 2026-09-22.

The gate only uses information that would have been available at the time, and it switches off when a metric is missing. Costs come from the positions the gated strategy actually holds. The thresholds are standard conventions, and I can't show they were fixed before I saw the test period.

In [1]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config
from sv import db
con = db.connect(read_only=True)


In [2]:
pd.read_csv(ROOT/'reports/oot_gate.csv')

,model,CAGR ungated / gated,max DD ungated / gated,Sharpe ungated / gated,off
0,momentum,37.8% / 26.8%,-28.6% / -28.2%,0.99 / 0.91,33.6%
1,gbm,10.5% / 0.0%,-37.1% / 0.0%,0.47 / n/a,100.0%
2,gbm_expected,18.4% / 3.5%,-34.5% / -21.5%,0.61 / 0.30,85.9%
3,gbm_weekly,22.9% / 1.9%,-39.6% / -36.3%,0.71 / 0.21,61.0%
4,clam_2021,17.2% / 0.0%,-32.4% / 0.0%,0.63 / n/a,100.0%
5,clam_weekly_cs_demeaned,-4.1% / 0.0%,-50.6% / 0.0%,0.01 / n/a,100.0%
6,clam_weekly_cs_rank_small,10.3% / 0.0%,-27.8% / 0.0%,0.52 / n/a,100.0%
7,clam_weekly_cs_rank_small_n94_seed20260922,9.2% / 0.0%,-49.8% / 0.0%,0.42 / n/a,100.0%
8,clam_weekly_cs_rank_small_n500_seed20260922,11.0% / 0.0%,-32.8% / 0.0%,0.55 / n/a,100.0%
9,clam_weekly_cs_rank_small_n3000_seed20260922,-0.3% / 0.0%,-25.5% / 0.0%,0.06 / n/a,100.0%


In [3]:
db.read(con, 'SELECT model,reason,COUNT(*) AS weeks FROM gate_decisions WHERE NOT gate_on AND date >= $start AND date <= $end GROUP BY model,reason ORDER BY model,weeks DESC', {'start':config.OOT_START,'end':config.EVALUATION_END})

,model,reason,weeks
0,clam_2021,"rolling_sharpe,auc_1w,active_drawdown",116
1,clam_2021,"auc_1w,active_drawdown",65
2,clam_2021,active_drawdown,56
3,clam_2021,"rolling_sharpe,active_drawdown",6
4,clam_weekly_cs_demeaned,"rolling_sharpe,auc_1w,active_drawdown",200
...,...,...,...
57,momentum,"psi_score,rolling_sharpe,auc_1w,active_drawdown",7
58,momentum,"psi_score,active_drawdown",4
59,momentum,rolling_sharpe,2
60,momentum,"psi_score,auc_1w,active_drawdown",2


In [4]:
con.close()